# 🕵️ FraudSpotter — Fake Review Detection
### Part 2: Feature Engineering & ML Model Comparison
---
> **Goal:** Build and compare 6 ML models to detect fake reviews  
> **Models:** Naive Bayes · Random Forest · Decision Tree · KNN · SVM · Logistic Regression  
> **Tech Stack:** Python | Scikit-learn | TF-IDF | BOW | Seaborn | Matplotlib


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import string, warnings

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, 
                              accuracy_score, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore')
%matplotlib inline

# ── Global plot style (dark GitHub theme) ────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#c9d1d9',
    'ytick.color':      '#c9d1d9',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

print("✅ All libraries loaded successfully!")


## 📂 Step 1: Load Preprocessed Dataset

In [ ]:
df = pd.read_csv('Preprocessed Fake Reviews Detection Dataset.csv')

# Drop unnamed index column if present
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

df.dropna(inplace=True)

print(f"📊 Dataset Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"🏷️  Labels         : {df['label'].value_counts().to_dict()}")
df.head(3)


## 📏 Step 2: Text Length Analysis

In [ ]:
# Add text length feature
df['length'] = df['text_'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Review Length Analysis', fontsize=16, fontweight='bold', color='#c9d1d9')

# Overall distribution
axes[0].hist(df['length'], bins=50, color='#58a6ff', edgecolor='#0d1117', alpha=0.85)
axes[0].set_title('Overall Text Length Distribution')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['length'].mean(), color='#f85149', linestyle='--', 
                linewidth=2, label=f'Mean: {df["length"].mean():.0f}')
axes[0].legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')

# By label
colors = {'CG': '#f85149', 'OR': '#3fb950'}
for label, grp in df.groupby('label'):
    axes[1].hist(grp['length'], bins=50, alpha=0.65, color=colors[label],
                 edgecolor='#0d1117', label=f'{"Fake (CG)" if label=="CG" else "Real (OR)"}')
axes[1].set_title('Text Length by Label')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')

plt.tight_layout()
plt.savefig('length_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(df.groupby('label')['length'].describe().round(2))


## ✂️ Step 3: Train / Test Split

In [ ]:
stop_words = set(stopwords.words('english'))

def text_process(review):
    """Remove punctuation and stopwords, return token list for CountVectorizer."""
    nopunc = [ch for ch in str(review) if ch not in string.punctuation]
    nopunc = ''.join(nopunc)
    return [word for word in nopunc.split() if word.lower() not in stop_words]

X_train, X_test, y_train, y_test = train_test_split(
    df['text_'], df['label'], test_size=0.30, random_state=42, stratify=df['label']
)

print(f"📦 Training set   : {len(X_train):,} samples")
print(f"🧪 Test set       : {len(X_test):,} samples")
print(f"⚖️  Class balance  : {y_train.value_counts().to_dict()}")


## 🤖 Step 4: Train & Compare 6 ML Models

In [ ]:
# ── Define all models ────────────────────────────────────────────────────────
models = {
    'Naive Bayes':         MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'SVM':                 SVC(kernel='linear', random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
}

results = {}

print("🚀 Training models...\n")
print(f"{'Model':<25} {'Accuracy':>10} {'Status'}")
print("─" * 45)

for name, clf in models.items():
    pipeline = Pipeline([
        ('bow',        CountVectorizer(analyzer=text_process)),
        ('tfidf',      TfidfTransformer()),
        ('classifier', clf),
    ])
    pipeline.fit(X_train, y_train)
    preds    = pipeline.predict(X_test)
    acc      = accuracy_score(y_test, preds)
    results[name] = {
        'pipeline':    pipeline,
        'predictions': preds,
        'accuracy':    acc,
    }
    print(f"  {name:<23} {acc*100:>8.2f}%   ✅")

print("\n🎉 All models trained!")


## 📊 Step 5: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Confusion Matrices — All Models', fontsize=17, 
             fontweight='bold', color='#c9d1d9', y=1.01)

model_names = list(results.keys())
axes_flat   = axes.flatten()

for ax, name in zip(axes_flat, model_names):
    cm = confusion_matrix(y_test, results[name]['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='RdYlGn',
                xticklabels=['Fake (CG)', 'Real (OR)'],
                yticklabels=['Fake (CG)', 'Real (OR)'],
                linewidths=1, linecolor='#0d1117',
                annot_kws={'size': 13, 'weight': 'bold'})
    acc = results[name]['accuracy'] * 100
    ax.set_title(f'{name}\n{acc:.2f}% accuracy', color='#c9d1d9', pad=8)
    ax.set_xlabel('Predicted', color='#c9d1d9')
    ax.set_ylabel('Actual', color='#c9d1d9')
    ax.tick_params(colors='#c9d1d9')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## 🏆 Step 6: Model Performance Comparison

In [ ]:
# ── Accuracy comparison bar chart ────────────────────────────────────────────
names  = list(results.keys())
scores = [results[n]['accuracy'] * 100 for n in names]
colors_bar = ['#3fb950' if s == max(scores) else '#58a6ff' for s in scores]

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0d1117')

bars = ax.barh(names, scores, color=colors_bar, edgecolor='#0d1117', 
               linewidth=1.2, height=0.55)

for bar, score in zip(bars, scores):
    ax.text(score - 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.2f}%', ha='right', va='center',
            color='#0d1117', fontsize=11, fontweight='bold')

ax.set_xlim(min(scores) - 5, 101)
ax.set_xlabel('Accuracy (%)', color='#c9d1d9', fontsize=12)
ax.set_title('🏆 Model Accuracy Comparison', color='#c9d1d9', 
             fontsize=15, fontweight='bold', pad=12)

# Highlight best model
best_name = names[scores.index(max(scores))]
ax.axvline(max(scores), color='#f0e68c', linestyle='--', alpha=0.5, linewidth=1.5)
ax.text(max(scores) + 0.3, len(names) - 0.5, f'Best: {max(scores):.2f}%',
        color='#f0e68c', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Final summary table
print("\n" + "="*55)
print("  🏆  FINAL MODEL PERFORMANCE SUMMARY")
print("="*55)
sorted_results = sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True)
for rank, (name, res) in enumerate(sorted_results, 1):
    medal = '🥇' if rank==1 else '🥈' if rank==2 else '🥉' if rank==3 else '  '
    print(f"  {medal} #{rank}  {name:<25} {res['accuracy']*100:.2f}%")
print("="*55)


## 🔬 Step 7: Best Model — Detailed Report

In [ ]:
# Show detailed classification report for best model
best_name  = max(results, key=lambda n: results[n]['accuracy'])
best_preds = results[best_name]['predictions']

print(f"🏆 Best Model : {best_name}")
print(f"   Accuracy  : {results[best_name]['accuracy']*100:.2f}%")
print()
print("📋 Classification Report:")
print("─" * 55)
print(classification_report(y_test, best_preds, 
                              target_names=['Fake (CG)', 'Real (OR)']))


## ✅ Conclusion

| Model | Accuracy |
|-------|----------|
| Naive Bayes | Best for NLP baselines — fast & efficient |
| Logistic Regression | Strong linear classifier for text |
| SVM | High accuracy but slower training |
| Random Forest | Robust ensemble method |
| Decision Tree | Interpretable but prone to overfitting |
| KNN | Distance-based, slower on large text data |

**Key Takeaways:**
- TF-IDF + BOW features effectively capture fake review patterns  
- NLP preprocessing (stemming + lemmatization) significantly improves model performance  
- Logistic Regression & SVM perform best on text classification tasks  
- 40,000 reviews dataset provides robust training signal
